## Preprocessing of the data: 

In [2]:
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

FOLDER_PATH = "dataset/MIA_SDG_Exercise"
SYNTHETIC_FILES = [f"synthetic_data{i}.csv" for i in range(1, 5)]
TEST_FILES = ["test_data_with_outliers.csv", "test_data_wto_outliers.csv"]
LABEL_COLUMN = "is_member"  # Only in test datasets

def load_dataset(path, drop_label=False):
    df = pd.read_csv(path)
    if drop_label and LABEL_COLUMN in df.columns:
        df = df.drop(columns=[LABEL_COLUMN])
    return df

def build_preprocessing_pipeline(df):
    categorical_cols = df.select_dtypes(include="object").columns.tolist()
    numeric_cols = df.select_dtypes(exclude="object").columns.tolist()

    # Treat 'readmission_status' as categorical if misclassified
    if "readmission_status" in numeric_cols:
        numeric_cols.remove("readmission_status")
        categorical_cols.append("readmission_status")

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=-999)),
        ("scaler", StandardScaler())
    ])

    preprocessor = ColumnTransformer([
        ("cat", cat_pipe, categorical_cols),
        ("num", num_pipe, numeric_cols)
    ])

    return preprocessor, categorical_cols + numeric_cols

def preprocess_pair(synth_df, test_df):
    # Find overlapping columns only
    shared_columns = [col for col in test_df.columns if col in synth_df.columns]
    synth_df = synth_df[shared_columns]
    test_df = test_df[shared_columns]

    # Build preprocessing pipeline on test set
    preprocessor, _ = build_preprocessing_pipeline(test_df)

    X_test = preprocessor.fit_transform(test_df)
    X_synth = preprocessor.transform(synth_df)

    return X_test, X_synth

def preprocess_all():
    results = {}
    for synth_file in SYNTHETIC_FILES:
        synth_path = os.path.join(FOLDER_PATH, synth_file)
        synth_df = load_dataset(synth_path)

        for test_file in TEST_FILES:
            test_path = os.path.join(FOLDER_PATH, test_file)
            test_df = load_dataset(test_path, drop_label=True)

            key = (synth_file, test_file)
            try:
                X_test, X_synth = preprocess_pair(synth_df.copy(), test_df.copy())
                results[key] = (X_test, X_synth)
            except Exception as e:
                print(f"[ERROR] Failed to preprocess {key}: {e}")

    return results  # dict[(synth_file, test_file)] = (X_test, X_synth)

if __name__ == "__main__":
    data = preprocess_all()
    print(f"✅ Preprocessed {len(data)} dataset pairs successfully.")

✅ Preprocessed 8 dataset pairs successfully.


## Attack 1

### **1. How it uses the given synthetic data? How does it process it?**

The synthetic data files (`synthetic_data1.csv` to `synthetic_data4.csv`) are generated samples meant to mimic real-world private data. These files are used to test if an attacker can **reverse-engineer membership**—i.e., guess if a specific record from the original dataset was used during training.

#### Processing steps:

* Each synthetic file is **loaded** using `pd.read_csv`.
* The corresponding **test files** (`test_data_with_outliers.csv` and `test_data_wto_outliers.csv`) are also loaded. These test files contain actual data samples, some of which are labeled as **`is_member = yes`** (they were used during synthetic data training), and others as **`is_member = no`** (they were not).
* The script **aligns the columns** between synthetic and test datasets to ensure a fair comparison.
* For each record in the test data (row-by-row), the script performs two attacks by comparing the test row to the synthetic data distribution:

  * Mode Collapse Attack
  * Conditional Imbalance Attack

---

### **2. Against what does it compare the synthetic data metrics to make predictions?**

The **goal** is to compute **anomaly scores** for each test row to determine if it was part of the training data (member) or not.

For each test row, the comparison is made between:

#### 🧪 **The test record itself** (1 row of real data)

vs.
🎲 **The full synthetic dataset** (multiple rows generated by the model)

The attack does **not** compare test rows directly against each other. Instead, it:

* Takes a **single test record**, computes some statistics,
* Compares those statistics to the **entire synthetic dataset**, and
* Derives a **difference score** (how “unusual” the test row is relative to the synthetic distribution).

The **intuition**: if a record is *too similar* to the synthetic distribution, it likely influenced the model ⇒ **member**. If it differs a lot, it probably wasn’t seen by the model ⇒ **non-member**.

These scores are then used to classify each test record as member/non-member by thresholding (median is used as default threshold), and attack performance is measured using:

* Accuracy, Precision, Recall, F1, AUC, ASR (Attack Success Rate).

---

### **3. Extremely simple overview of the attack**

This attack answers the question:

> "Can I tell if a specific real person’s data was used to train this synthetic data generator?"

#### In simpler terms:

* You have fake data generated by a model (synthetic).
* You also have real data, but you're not sure which real samples were used to generate the fake ones.
* For each real sample, you:

  1. Check **how well it fits** the synthetic data.
  2. If it fits **too well**, it's likely in the training data.
  3. If it looks **too different**, it's probably not in the training data.

By repeating this across many samples and measuring how accurate your guesses are, you perform a **membership inference attack**.

---

### Attack Details

#### **Attack 1: Mode Collapse**

* For each **categorical column**, it compares:

  * The **entropy** (diversity) and **distinct ratio** (how many unique values) of that column in the **synthetic dataset** and the **single test row**.
  * If the test row’s categorical features have unusually low diversity compared to the synthetic data (or too similar), that’s suspicious.
  * This detects **mode collapse** — a phenomenon where generative models output only a few repeated modes (e.g., same provider or diagnosis).

#### **Attack 2: Conditional Imbalance**

* For certain **condition columns** (e.g., `drug_013`, `hba1c_result`) and **numeric targets** (e.g., `lab_test_count`, `stay_duration_days`):

  * It finds **rare values** (values that occur <20% in the real test data).
  * Then it compares the **distribution** of the numeric targets **conditioned on those rare values** using the **KS-test** (statistical test for distribution difference).
  * If the conditional distribution of a test row matches the synthetic data well (or differs a lot), it might indicate its membership.

In [12]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from scipy.stats import entropy, ks_2samp

FOLDER_PATH = "dataset/MIA_SDG_Exercise"
SYNTHETIC_FILES = [f"synthetic_data{i}.csv" for i in range(1, 5)]
TEST_FILES = ["test_data_with_outliers.csv", "test_data_wto_outliers.csv"]
LABEL_COLUMN = "is_member"

CATEGORICAL_FEATURES = ["secondary_diagnosis", "provider_specialty", "insurance_type_code"]
CONDITION_COLUMNS = ["drug_013", "hba1c_result"]
NUMERIC_TARGETS = ["lab_test_count", "stay_duration_days"]

# --- Helper functions --- #
def safe_entropy(series):
    if series.empty:
        return 0.0
    counts = series.value_counts(normalize=True)
    return entropy(counts, base=2)

def distinct_ratio(series):
    if len(series) == 0:
        return 0.0
    return series.nunique() / len(series)


def compute_mode_collapse_score(synth_df, test_df):
    scores = []
    for col in CATEGORICAL_FEATURES:
        if col not in synth_df.columns or col not in test_df.columns:
            continue
        e_test = safe_entropy(test_df[col].dropna())
        e_synth = safe_entropy(synth_df[col].dropna())
        d_test = distinct_ratio(test_df[col].dropna())
        d_synth = distinct_ratio(synth_df[col].dropna())
        # Difference in entropy and distinct ratio
        score = abs(e_test - e_synth) + abs(d_test - d_synth)
        scores.append(score)
    return np.mean(scores) if scores else 0.0

def compute_conditional_imbalance_score(synth_df, test_df):
    scores = []
    for cond_col in CONDITION_COLUMNS:
        for target in NUMERIC_TARGETS:
            if cond_col not in synth_df.columns or target not in synth_df.columns:
                continue
            rare_values = test_df[cond_col].value_counts(normalize=True)
            rare_values = rare_values[rare_values < 0.2].index.tolist()
            for val in rare_values:
                synth_subset = synth_df[synth_df[cond_col] == val][target].dropna()
                test_subset = test_df[test_df[cond_col] == val][target].dropna()
                if len(synth_subset) > 5 and len(test_subset) > 5:
                    stat, _ = ks_2samp(test_subset, synth_subset)
                    scores.append(stat)
    return np.mean(scores) if scores else 0.0

# --- Attack Evaluation --- #
def evaluate_attack(scores, labels, threshold=None):
    scores = np.array(scores)
    true_labels = np.array([1 if l == "yes" else 0 for l in labels])

    if threshold is None:
        threshold = np.median(scores)  # simple decision boundary

    preds = (scores >= threshold).astype(int)

    metrics = {
        "Accuracy": accuracy_score(true_labels, preds),
        "Precision": precision_score(true_labels, preds, zero_division=0),
        "Recall": recall_score(true_labels, preds, zero_division=0),
        "F1": f1_score(true_labels, preds, zero_division=0),
        "AUC": roc_auc_score(true_labels, scores),
        "ASR": float(np.mean(preds == true_labels))
    }
    return metrics

# --- Main Attack Runner --- #
def run_attacks():
    results = defaultdict(dict)

    for synth_file in SYNTHETIC_FILES:
        synth_path = os.path.join(FOLDER_PATH, synth_file)
        synth_df = pd.read_csv(synth_path)

        for test_file in TEST_FILES:
            test_path = os.path.join(FOLDER_PATH, test_file)
            test_df = pd.read_csv(test_path)

            key = f"{synth_file} vs {test_file}"
            print(f"\n🔍 Running attacks on: {key}")

            # Align columns
            common_cols = [col for col in test_df.columns if col in synth_df.columns and col != LABEL_COLUMN]
            test_subset = test_df[common_cols + [LABEL_COLUMN]].dropna(subset=[LABEL_COLUMN])
            synth_subset = synth_df[common_cols].copy()

            test_real = test_subset[test_subset[LABEL_COLUMN] == "yes"].drop(columns=[LABEL_COLUMN])
            test_nonreal = test_subset[test_subset[LABEL_COLUMN] == "no"].drop(columns=[LABEL_COLUMN])
            all_labels = test_subset[LABEL_COLUMN].tolist()

            # Attack 1: Mode Collapse
            mc_scores = []
            for _, row in test_subset.iterrows():
                single_row = pd.DataFrame([row.drop(labels=[LABEL_COLUMN])])
                score = compute_mode_collapse_score(synth_subset, single_row)
                mc_scores.append(score)

            mc_metrics = evaluate_attack(mc_scores, all_labels)
            results[key]["Mode Collapse"] = mc_metrics

            # Attack 2: Conditional Imbalance
            ci_scores = []
            for _, row in test_subset.iterrows():
                single_row = pd.DataFrame([row.drop(labels=[LABEL_COLUMN])])
                score = compute_conditional_imbalance_score(synth_subset, single_row)
                ci_scores.append(score)

            ci_metrics = evaluate_attack(ci_scores, all_labels)
            results[key]["Conditional Imbalance"] = ci_metrics

    return results

# --- Print Results --- #
def print_results(results):
    for pair, attacks in results.items():
        print(f"\n==== Results for {pair} ====")
        for attack_name, metrics in attacks.items():
            print(f"Attack: {attack_name}")
            for metric, val in metrics.items():
                print(f"{metric:>10}: {val:.4f}")
            print("-" * 30)

if __name__ == "__main__":
    attack_results = run_attacks()
    print_results(attack_results)



🔍 Running attacks on: synthetic_data1.csv vs test_data_with_outliers.csv

🔍 Running attacks on: synthetic_data1.csv vs test_data_wto_outliers.csv

🔍 Running attacks on: synthetic_data2.csv vs test_data_with_outliers.csv

🔍 Running attacks on: synthetic_data2.csv vs test_data_wto_outliers.csv

🔍 Running attacks on: synthetic_data3.csv vs test_data_with_outliers.csv

🔍 Running attacks on: synthetic_data3.csv vs test_data_wto_outliers.csv

🔍 Running attacks on: synthetic_data4.csv vs test_data_with_outliers.csv

🔍 Running attacks on: synthetic_data4.csv vs test_data_wto_outliers.csv

==== Results for synthetic_data1.csv vs test_data_with_outliers.csv ====
Attack: Mode Collapse
  Accuracy: 0.4479
 Precision: 0.4658
    Recall: 0.7083
        F1: 0.5620
       AUC: 0.4559
       ASR: 0.4479
------------------------------
Attack: Conditional Imbalance
  Accuracy: 0.5000
 Precision: 0.5000
    Recall: 1.0000
        F1: 0.6667
       AUC: 0.5000
       ASR: 0.5000
---------------------------

## Attack 2

This script performs a **Conditional Imbalance Membership Inference Attack** (MIA), specifically targeting synthetic datasets. Its goal is to determine whether a specific test record was used to train the synthetic data generator (i.e., whether the record is a **member** of the training data). Below is a detailed breakdown of how the attack works, following your requested structure.

---

### **1. How it uses the given synthetic data? How does it process it?**

The synthetic data files (`synthetic_data1.csv` to `synthetic_data4.csv`) represent artificially generated data that mimics real data distributions.

The processing steps are as follows:

* Each synthetic file is **loaded** using pandas (`pd.read_csv`).
* The **test files** (`test_data_with_outliers.csv`, `test_data_wto_outliers.csv`) are also loaded. These contain real samples, some labeled with `is_member = yes` (used in training the generator) and others with `is_member = no` (not used).
* For each test record, the attack:

  * Drops the `is_member` label.
  * Looks at specific **conditional columns** (e.g., `drug_013`, `hba1c_result`).
  * For each condition value (e.g., `hba1c_result = 'High'`), it **filters** the synthetic data to get matching records.
  * Within that subset, it compares the **numeric features** (`lab_test_count`, `stay_duration_days`) of the synthetic records to the value in the test record.

This is done for **each row** in the test set, so the synthetic dataset is repeatedly queried and filtered based on the conditions.

---

### **2. Against what does it compare the synthetic data metrics, values, or parameters to make predictions?**

The attack compares **each individual test row** against the **synthetic data distribution** that matches on the conditional feature.

Let’s make this concrete:

* Suppose a test row has `hba1c_result = "High"` and `lab_test_count = 4`.
* The attack:

  * Finds all synthetic rows where `hba1c_result == "High"`.
  * Extracts all their `lab_test_count` values.
  * Compares the test value (4) against the synthetic distribution using the **Kolmogorov–Smirnov (KS) statistic**, which measures the difference between distributions.

This is repeated for:

* Every conditional column (`drug_013`, `hba1c_result`)
* Every numeric target column (`lab_test_count`, `stay_duration_days`)

The KS statistic values are averaged to produce a **final score** for each test row. This score indicates how closely that test record resembles the synthetic distribution. A **low score** means the test row matches the synthetic distribution well (possible member), and a **high score** means it doesn’t match well (likely non-member).

Predictions are made by thresholding these scores:

* If the score is **lower than the median**, classify as member.
* If it's **higher**, classify as non-member.

The predicted labels are then evaluated against the true labels using:

* Accuracy, Precision, Recall, F1 Score, AUC (ROC), and ASR (Attack Success Rate).

---

### **3. Extremely simple overview of the attack**

This attack checks:

> "Does this test record *fit too well* with the synthetic data when we look at specific conditions and outcomes?"

If it fits the synthetic data *too closely* under certain conditions (e.g., "patients who took drug\_013 = 1 have stay\_duration\_days = 5"), it’s probably a **member** — i.e., the synthetic model saw it during training.

So for every test row, the attack:

* Looks at conditional values like drug prescriptions or test results.
* Filters the synthetic data by these conditions.
* Compares the test value against that subset using a statistical test (KS-test).
* Scores how much the test row stands out.
* Guesses if it was in the training set (member) or not based on that score.

In [3]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score
)
from scipy.stats import ks_2samp

FOLDER_PATH = "dataset/MIA_SDG_Exercise"
SYNTHETIC_FILES = [f"synthetic_data{i}.csv" for i in range(1, 5)]
TEST_FILES = ["test_data_with_outliers.csv", "test_data_wto_outliers.csv"]
LABEL_COLUMN = "is_member"

# Attack config
CONDITION_COLUMNS = ["drug_013", "hba1c_result"]
NUMERIC_TARGETS = ["lab_test_count", "stay_duration_days"]

def load_dataset(path, drop_label=False):
    df = pd.read_csv(path)
    if drop_label and LABEL_COLUMN in df.columns:
        df = df.drop(columns=[LABEL_COLUMN])
    return df

def compute_conditional_imbalance_score(synth_df, test_row, cond_cols, num_targets):
    scores = []
    for cond in cond_cols:
        if cond not in synth_df.columns or cond not in test_row:
            continue
        cond_val = test_row[cond]
        matched_synth = synth_df[synth_df[cond] == cond_val]
        if matched_synth.empty:
            continue

        for target in num_targets:
            if target in matched_synth.columns and target in test_row:
                synth_values = matched_synth[target].dropna()
                test_val = test_row[target]

                if pd.notna(test_val) and len(synth_values) > 5:
                    ks_stat, _ = ks_2samp(synth_values, [test_val] * len(synth_values))
                    scores.append(ks_stat)
    return np.mean(scores) if scores else 0.0

def evaluate_attack(scores, labels, threshold=None):
    scores = np.array(scores)
    true_labels = np.array([1 if l == "yes" else 0 for l in labels])

    if threshold is None:
        threshold = np.median(scores)

    preds = (scores >= threshold).astype(int)

    metrics = {
        "Accuracy": accuracy_score(true_labels, preds),
        "Precision": precision_score(true_labels, preds, zero_division=0),
        "Recall": recall_score(true_labels, preds, zero_division=0),
        "F1": f1_score(true_labels, preds, zero_division=0),
        "AUC": roc_auc_score(true_labels, scores),
        "ASR": float(np.mean(preds == true_labels))
    }
    return metrics

def run_conditional_imbalance_attack():
    results = {}

    for synth_file in SYNTHETIC_FILES:
        synth_path = os.path.join(FOLDER_PATH, synth_file)
        synth_df = pd.read_csv(synth_path)

        for test_file in TEST_FILES:
            test_path = os.path.join(FOLDER_PATH, test_file)
            test_df = pd.read_csv(test_path)

            if LABEL_COLUMN not in test_df.columns:
                print(f"⚠️ Skipping {test_file}: No '{LABEL_COLUMN}' column.")
                continue

            common_cols = [col for col in test_df.columns if col in synth_df.columns and col != LABEL_COLUMN]
            test_df = test_df[common_cols + [LABEL_COLUMN]]
            synth_df = synth_df[common_cols]

            scores = []
            labels = test_df[LABEL_COLUMN].tolist()

            for _, row in test_df.iterrows():
                test_row = row.drop(labels=[LABEL_COLUMN])
                score = compute_conditional_imbalance_score(synth_df, test_row, CONDITION_COLUMNS, NUMERIC_TARGETS)
                scores.append(score)

            metrics = evaluate_attack(scores, labels)
            key = f"{synth_file} vs {test_file}"
            results[key] = metrics

    return results

def print_results(results):
    for pair, metrics in results.items():
        print(f"\n📊 Results for {pair}")
        for metric, val in metrics.items():
            print(f"{metric:>10}: {val:.4f}")
        print("-" * 40)

if __name__ == "__main__":
    attack_results = run_conditional_imbalance_attack()
    print_results(attack_results)



📊 Results for synthetic_data1.csv vs test_data_with_outliers.csv
  Accuracy: 0.6042
 Precision: 0.6042
    Recall: 0.6042
        F1: 0.6042
       AUC: 0.6801
       ASR: 0.6042
----------------------------------------

📊 Results for synthetic_data1.csv vs test_data_wto_outliers.csv
  Accuracy: 0.4000
 Precision: 0.4000
    Recall: 0.4000
        F1: 0.4000
       AUC: 0.3800
       ASR: 0.4000
----------------------------------------

📊 Results for synthetic_data2.csv vs test_data_with_outliers.csv
  Accuracy: 0.6458
 Precision: 0.6458
    Recall: 0.6458
        F1: 0.6458
       AUC: 0.6888
       ASR: 0.6458
----------------------------------------

📊 Results for synthetic_data2.csv vs test_data_wto_outliers.csv
  Accuracy: 0.3500
 Precision: 0.3500
    Recall: 0.3500
        F1: 0.3500
       AUC: 0.4225
       ASR: 0.3500
----------------------------------------

📊 Results for synthetic_data3.csv vs test_data_with_outliers.csv
  Accuracy: 0.6458
 Precision: 0.6458
    Recall: 0.

## Attack 3
This script implements a **Gaussianity-based Membership Inference Attack (MIA)** on synthetic data. It tries to infer whether a real data record was used in training the synthetic data generator, based on how "normal" (in the statistical sense) the generated numeric feature distributions are.

Let’s explain it thoroughly using your three requested points:

---

### **1. How it uses the given synthetic data? How does it process it?**

The script uses the synthetic datasets (`synthetic_data1.csv` to `synthetic_data4.csv`) as the **output** of a generative model trained on private (real) data. It assumes that these synthetic datasets **approximate the distributions** of the real training records.

Here’s how it processes them:

* Loads each synthetic dataset (`synth_df`) using `pandas.read_csv`.
* Loads corresponding real-world test data (`test_df`), which contains a label `is_member` marking whether each record was used to train the generator.
* Aligns columns between `synth_df` and `test_df`, removing the label from `synth_df`.
* For **each row** in the test data:

  * Drops the `is_member` label to isolate the features.
  * Uses only the numeric columns.
  * For each numeric column, it:

    * Pulls the column’s values from the synthetic dataset (i.e., the distribution).
    * Runs **normality tests** (Shapiro-Wilk and Anderson-Darling) on that distribution.
    * Uses the test result statistics to compute a **"Gaussianity score"** — a numeric measure of how normally distributed the synthetic data looks in that column.
  * Averages these scores across all numeric columns.

This gives a **score per test row** reflecting how "Gaussian" the synthetic data looks for the features of that row.

---

### **2. Against what does it compare the synthetic data metrics, values, or parameters to make predictions?**

The attack doesn’t compare one test row directly against other test rows. Instead, it compares **how well the synthetic data fits a normal distribution** for the numeric features present in the test row.

Here’s the key logic:

* For each test row, we compute a Gaussianity score based on **the synthetic distribution** of numeric features relevant to that row.
* The **core idea**: If the synthetic generator overfit some training data, its generated values will **look less like Gaussian noise** and more like sharp memorized patterns. So the **less Gaussian** the synthetic feature distributions appear (especially in columns where the test row has a value), the more likely it is that the test row **was used during training**.
* These scores are collected for all test rows and thresholded (median by default) to classify them as **"yes" (member)** or **"no" (non-member)**.

Predictions are evaluated using:

* Accuracy, Precision, Recall, F1 score, AUC, and ASR (Attack Success Rate).

---

### **3. Extremely simple overview of the attack**

Think of it like this:

> "If the synthetic data is **not very noisy** and seems to follow **non-Gaussian patterns**, it probably reflects the real data it was trained on."

So for each real test record:

* The attacker looks at the synthetic values for each numeric column.
* Measures how "Gaussian" (normally distributed) these synthetic values are.
* If the synthetic values **deviate from a normal shape**, that could mean the generator memorized specific values.
* If this happens in columns related to the test record, it may mean the record was **in the training set** (i.e., a **member**).

The attack **doesn’t look at the test row’s actual values** beyond using them to decide which columns to analyze. Instead, it checks how "overfit" the generator appears, assuming training data would make the synthetic output **less Gaussian**.

---

### 🔍 Bonus: Why this attack can work

Generative models like CTGAN or Gaussian copulas often try to add randomness to mimic real data. But if they accidentally memorize the training data, the generated distributions become:

* Too sharp
* Too flat
* Not noisy enough

These flaws can be caught using statistical tests for **normality**, and that’s what this attack exploits.

Let me know if you want to visualize how Shapiro and Anderson behave on synthetic distributions or want to combine this with other attacks like mode collapse or conditional imbalance.


In [14]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from scipy.stats import shapiro, anderson

FOLDER_PATH = "dataset/MIA_SDG_Exercise"
SYNTHETIC_FILES = [f"synthetic_data{i}.csv" for i in range(1, 5)]
TEST_FILES = ["test_data_with_outliers.csv", "test_data_wto_outliers.csv"]
LABEL_COLUMN = "is_member"

NUMERIC_COLUMNS = [
    "lab_test_count", "medication_count", "stay_duration_days", "inpatient_visits",
    "diagnosis_count", "procedure_count", "entry_type_code", "entry_origin_code",
    "outpatient_visits", "emergency_visits", "exit_status_code"
]

def load_dataset(path, drop_label=False):
    df = pd.read_csv(path)
    if drop_label and LABEL_COLUMN in df.columns:
        df = df.drop(columns=[LABEL_COLUMN])
    return df

def normality_score(values):
    values = np.array(values.dropna())
    if len(values) < 5:
        return 0.0

    try:
        shap_stat, _ = shapiro(values[:5000])  # Shapiro-Wilk (limit for speed)
        and_stat = anderson(values).statistic
        return (1 - shap_stat) + (and_stat / 20)  # lower = more normal-like
    except:
        return 0.0

def compute_gaussianity_score(synth_df, test_row, numeric_cols):
    scores = []
    for col in numeric_cols:
        if col not in synth_df.columns or col not in test_row.index:
            continue
        val = test_row[col]
        if pd.isna(val):
            continue
        series = synth_df[col].dropna()
        if len(series) < 10:
            continue
        score = normality_score(series)
        scores.append(score)
    return np.mean(scores) if scores else 0.0

def evaluate_attack(scores, labels, threshold=None):
    scores = np.array(scores)
    true_labels = np.array([1 if l == "yes" else 0 for l in labels])

    if threshold is None:
        threshold = np.median(scores)  # simple threshold
    preds = (scores >= threshold).astype(int)

    metrics = {
        "Accuracy": accuracy_score(true_labels, preds),
        "Precision": precision_score(true_labels, preds, zero_division=0),
        "Recall": recall_score(true_labels, preds, zero_division=0),
        "F1": f1_score(true_labels, preds, zero_division=0),
        "AUC": roc_auc_score(true_labels, scores),
        "ASR": float(np.mean(preds == true_labels))
    }
    return metrics

def run_gaussianity_attack():
    results = defaultdict(dict)

    for synth_file in SYNTHETIC_FILES:
        synth_path = os.path.join(FOLDER_PATH, synth_file)
        synth_df = pd.read_csv(synth_path)

        for test_file in TEST_FILES:
            test_path = os.path.join(FOLDER_PATH, test_file)
            test_df = pd.read_csv(test_path)

            key = f"{synth_file} vs {test_file}"
            print(f"\n🔍 Running Gaussianity Attack on: {key}")

            common_cols = [col for col in test_df.columns if col in synth_df.columns and col != LABEL_COLUMN]
            test_subset = test_df[common_cols + [LABEL_COLUMN]].dropna(subset=[LABEL_COLUMN])
            synth_subset = synth_df[common_cols].copy()
            all_labels = test_subset[LABEL_COLUMN].tolist()

            g_scores = []
            for _, row in test_subset.iterrows():
                score = compute_gaussianity_score(synth_subset, row.drop(labels=[LABEL_COLUMN]), NUMERIC_COLUMNS)
                g_scores.append(score)

            metrics = evaluate_attack(g_scores, all_labels)
            results[key]["Gaussianity Score"] = metrics

    return results

def print_results(results):
    for pair, attacks in results.items():
        print(f"\n==== Results for {pair} ====")
        for attack_name, metrics in attacks.items():
            print(f"Attack: {attack_name}")
            for metric, val in metrics.items():
                print(f"{metric:>10}: {val:.4f}")
            print("-" * 30)

if __name__ == "__main__":
    attack_results = run_gaussianity_attack()
    print_results(attack_results)



🔍 Running Gaussianity Attack on: synthetic_data1.csv vs test_data_with_outliers.csv

🔍 Running Gaussianity Attack on: synthetic_data1.csv vs test_data_wto_outliers.csv

🔍 Running Gaussianity Attack on: synthetic_data2.csv vs test_data_with_outliers.csv

🔍 Running Gaussianity Attack on: synthetic_data2.csv vs test_data_wto_outliers.csv

🔍 Running Gaussianity Attack on: synthetic_data3.csv vs test_data_with_outliers.csv

🔍 Running Gaussianity Attack on: synthetic_data3.csv vs test_data_wto_outliers.csv

🔍 Running Gaussianity Attack on: synthetic_data4.csv vs test_data_with_outliers.csv

🔍 Running Gaussianity Attack on: synthetic_data4.csv vs test_data_wto_outliers.csv

==== Results for synthetic_data1.csv vs test_data_with_outliers.csv ====
Attack: Gaussianity Score
  Accuracy: 0.5000
 Precision: 0.5000
    Recall: 1.0000
        F1: 0.6667
       AUC: 0.5000
       ASR: 0.5000
------------------------------

==== Results for synthetic_data1.csv vs test_data_wto_outliers.csv ====
Attack

## Attack 4

In [15]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

FOLDER_PATH = "dataset/MIA_SDG_Exercise"
SYNTHETIC_FILES = [f"synthetic_data{i}.csv" for i in range(1, 5)]
TEST_FILES = ["test_data_with_outliers.csv", "test_data_wto_outliers.csv"]
LABEL_COLUMN = "is_member"

NUMERIC_COLUMNS = [
    "lab_test_count", "medication_count", "stay_duration_days", "inpatient_visits",
    "diagnosis_count", "procedure_count", "entry_type_code", "entry_origin_code",
    "outpatient_visits", "emergency_visits", "exit_status_code"
]

# --- Evaluation Helper --- #
def evaluate_attack(scores, labels, threshold=None):
    scores = np.array(scores)
    true_labels = np.array([1 if l == "yes" else 0 for l in labels])

    if threshold is None:
        threshold = np.median(scores)
    preds = (scores >= threshold).astype(int)

    metrics = {
        "Accuracy": accuracy_score(true_labels, preds),
        "Precision": precision_score(true_labels, preds, zero_division=0),
        "Recall": recall_score(true_labels, preds, zero_division=0),
        "F1": f1_score(true_labels, preds, zero_division=0),
        "AUC": roc_auc_score(true_labels, scores),
        "ASR": float(np.mean(preds == true_labels))
    }
    return metrics

# --- Main Score Function --- #
def compute_clustering_bias_score(synth_df, test_row, cols, n_clusters=2):
    # Combine test_row + synthetic sample space
    row_df = pd.DataFrame([test_row[cols]])
    df_combined = pd.concat([synth_df[cols], row_df], ignore_index=True).dropna()

    if df_combined.shape[0] < n_clusters + 1:
        return 0.0

    try:
        X = StandardScaler().fit_transform(df_combined.values)
        kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
        cluster_labels = kmeans.fit_predict(X)
        score = silhouette_score(X, cluster_labels)
        return score
    except Exception:
        return 0.0

# --- Attack Execution --- #
def run_reconstruction_bias_attack():
    results = defaultdict(dict)

    for synth_file in SYNTHETIC_FILES:
        synth_path = os.path.join(FOLDER_PATH, synth_file)
        synth_df = pd.read_csv(synth_path)

        for test_file in TEST_FILES:
            test_path = os.path.join(FOLDER_PATH, test_file)
            test_df = pd.read_csv(test_path)

            key = f"{synth_file} vs {test_file}"
            print(f"\n🔍 Running Reconstruction Bias Attack on: {key}")

            common_cols = [col for col in test_df.columns if col in synth_df.columns and col != LABEL_COLUMN]
            test_subset = test_df[common_cols + [LABEL_COLUMN]].dropna(subset=[LABEL_COLUMN])
            synth_subset = synth_df[common_cols].copy()

            all_labels = test_subset[LABEL_COLUMN].tolist()
            scores = []

            for _, row in test_subset.iterrows():
                score = compute_clustering_bias_score(synth_subset, row, NUMERIC_COLUMNS, n_clusters=2)
                scores.append(score)

            metrics = evaluate_attack(scores, all_labels)
            results[key]["Reconstruction Bias"] = metrics

    return results

# --- Print Results --- #
def print_results(results):
    for pair, attacks in results.items():
        print(f"\n==== Results for {pair} ====")
        for attack_name, metrics in attacks.items():
            print(f"Attack: {attack_name}")
            for metric, val in metrics.items():
                print(f"{metric:>10}: {val:.4f}")
            print("-" * 30)

if __name__ == "__main__":
    attack_results = run_reconstruction_bias_attack()
    print_results(attack_results)



🔍 Running Reconstruction Bias Attack on: synthetic_data1.csv vs test_data_with_outliers.csv

🔍 Running Reconstruction Bias Attack on: synthetic_data1.csv vs test_data_wto_outliers.csv

🔍 Running Reconstruction Bias Attack on: synthetic_data2.csv vs test_data_with_outliers.csv

🔍 Running Reconstruction Bias Attack on: synthetic_data2.csv vs test_data_wto_outliers.csv

🔍 Running Reconstruction Bias Attack on: synthetic_data3.csv vs test_data_with_outliers.csv

🔍 Running Reconstruction Bias Attack on: synthetic_data3.csv vs test_data_wto_outliers.csv

🔍 Running Reconstruction Bias Attack on: synthetic_data4.csv vs test_data_with_outliers.csv

🔍 Running Reconstruction Bias Attack on: synthetic_data4.csv vs test_data_wto_outliers.csv

==== Results for synthetic_data1.csv vs test_data_with_outliers.csv ====
Attack: Reconstruction Bias
  Accuracy: 0.8542
 Precision: 0.8542
    Recall: 0.8542
        F1: 0.8542
       AUC: 0.8490
       ASR: 0.8542
------------------------------

==== Results

## Attack 5

In [16]:
import os
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.metrics import (
    mutual_info_score,
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.preprocessing import LabelEncoder

FOLDER_PATH = "dataset/MIA_SDG_Exercise"
SYNTHETIC_FILES = [f"synthetic_data{i}.csv" for i in range(1, 5)]
TEST_FILES = ["test_data_with_outliers.csv", "test_data_wto_outliers.csv"]
LABEL_COLUMN = "is_member"

# Define feature pairs to evaluate dependency on
DEPENDENT_PAIRS = [
    ("primary_diagnosis", "procedure_count"),
    ("secondary_diagnosis", "exit_status_code"),
    ("age_range", "medication_count"),
    ("ethnic_group", "inpatient_visits")
]

def encode_categorical(series):
    """Converts a categorical column to numeric using label encoding."""
    le = LabelEncoder()
    return le.fit_transform(series.astype(str))

def compute_mutual_info(df, col_x, col_y):
    if col_x not in df.columns or col_y not in df.columns:
        return 0.0

    x = df[col_x].dropna()
    y = df[col_y].dropna()
    joined = pd.concat([x, y], axis=1).dropna()

    if joined.empty:
        return 0.0

    x_enc = encode_categorical(joined[col_x])
    y_enc = encode_categorical(joined[col_y])
    return mutual_info_score(x_enc, y_enc)

def compute_dependency_score(synth_df, test_row, pairs):
    test_df = pd.DataFrame([test_row])
    score_diffs = []

    for x, y in pairs:
        if x in synth_df.columns and y in synth_df.columns and x in test_df.columns and y in test_df.columns:
            mi_synth = compute_mutual_info(synth_df, x, y)
            mi_test = compute_mutual_info(test_df, x, y)
            diff = abs(mi_synth - mi_test)
            score_diffs.append(diff)

    return np.mean(score_diffs) if score_diffs else 0.0

def evaluate_attack(scores, labels, threshold=None):
    scores = np.array(scores)
    true_labels = np.array([1 if l == "yes" else 0 for l in labels])

    if threshold is None:
        threshold = np.median(scores)
    preds = (scores >= threshold).astype(int)

    metrics = {
        "Accuracy": accuracy_score(true_labels, preds),
        "Precision": precision_score(true_labels, preds, zero_division=0),
        "Recall": recall_score(true_labels, preds, zero_division=0),
        "F1": f1_score(true_labels, preds, zero_division=0),
        "AUC": roc_auc_score(true_labels, scores),
        "ASR": float(np.mean(preds == true_labels))
    }
    return metrics

def run_dependency_attack():
    results = defaultdict(dict)

    for synth_file in SYNTHETIC_FILES:
        synth_path = os.path.join(FOLDER_PATH, synth_file)
        synth_df = pd.read_csv(synth_path)

        for test_file in TEST_FILES:
            test_path = os.path.join(FOLDER_PATH, test_file)
            test_df = pd.read_csv(test_path)

            key = f"{synth_file} vs {test_file}"
            print(f"\n🔍 Running Overfitting Dependency Attack on: {key}")

            shared_cols = [col for col in test_df.columns if col in synth_df.columns and col != LABEL_COLUMN]
            test_subset = test_df[shared_cols + [LABEL_COLUMN]].dropna(subset=[LABEL_COLUMN])
            synth_subset = synth_df[shared_cols].copy()

            scores = []
            labels = test_subset[LABEL_COLUMN].tolist()

            for _, row in test_subset.iterrows():
                score = compute_dependency_score(synth_subset, row, DEPENDENT_PAIRS)
                scores.append(score)

            metrics = evaluate_attack(scores, labels)
            results[key]["Overfitting Dependency"] = metrics

    return results

def print_results(results):
    for pair, attacks in results.items():
        print(f"\n==== Results for {pair} ====")
        for attack_name, metrics in attacks.items():
            print(f"Attack: {attack_name}")
            for metric, val in metrics.items():
                print(f"{metric:>10}: {val:.4f}")
            print("-" * 30)

if __name__ == "__main__":
    attack_results = run_dependency_attack()
    print_results(attack_results)



🔍 Running Overfitting Dependency Attack on: synthetic_data1.csv vs test_data_with_outliers.csv

🔍 Running Overfitting Dependency Attack on: synthetic_data1.csv vs test_data_wto_outliers.csv

🔍 Running Overfitting Dependency Attack on: synthetic_data2.csv vs test_data_with_outliers.csv

🔍 Running Overfitting Dependency Attack on: synthetic_data2.csv vs test_data_wto_outliers.csv

🔍 Running Overfitting Dependency Attack on: synthetic_data3.csv vs test_data_with_outliers.csv

🔍 Running Overfitting Dependency Attack on: synthetic_data3.csv vs test_data_wto_outliers.csv

🔍 Running Overfitting Dependency Attack on: synthetic_data4.csv vs test_data_with_outliers.csv

🔍 Running Overfitting Dependency Attack on: synthetic_data4.csv vs test_data_wto_outliers.csv

==== Results for synthetic_data1.csv vs test_data_with_outliers.csv ====
Attack: Overfitting Dependency
  Accuracy: 0.5000
 Precision: 0.5000
    Recall: 1.0000
        F1: 0.6667
       AUC: 0.5000
       ASR: 0.5000
-----------------

## Attack 6

In [17]:
import os
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

FOLDER_PATH = "dataset/MIA_SDG_Exercise"
SYNTHETIC_FILES = [f"synthetic_data{i}.csv" for i in range(1, 5)]
TEST_FILE = "test_data_wto_outliers.csv"  # Use outlier-free test set
LABEL_COLUMN = "is_member"

# Define rare combinations to monitor
RARE_COMBOS = [
    {"med_change_flag": "decrease", "drug_008": "constant", "exit_status_code": 9},
    {"drug_015": "increase", "drug_006": "constant", "inpatient_visits": 4},
    {"drug_017": "decrease", "ethnic_group": "Asian", "provider_specialty": "Orthopedic Surgeon"},
]

def match_rare_combo(row, combo):
    for col, val in combo.items():
        if col not in row or pd.isna(row[col]):
            return False
        if row[col] != val:
            return False
    return True

def compute_combo_frequency(df, combo):
    return df.apply(lambda row: match_rare_combo(row, combo), axis=1).mean()

def compute_rare_combo_score(synth_df, test_df, combos):
    scores = []
    for combo in combos:
        synth_freq = compute_combo_frequency(synth_df, combo)
        test_freq = compute_combo_frequency(test_df, combo)
        scores.append(abs(synth_freq - test_freq))
    return np.mean(scores) if scores else 0.0

def evaluate_attack(scores, labels, threshold=None):
    scores = np.array(scores)
    true_labels = np.array([1 if l == "yes" else 0 for l in labels])

    if threshold is None:
        threshold = np.median(scores)
    preds = (scores >= threshold).astype(int)

    return {
        "Accuracy": accuracy_score(true_labels, preds),
        "Precision": precision_score(true_labels, preds, zero_division=0),
        "Recall": recall_score(true_labels, preds, zero_division=0),
        "F1": f1_score(true_labels, preds, zero_division=0),
        "AUC": roc_auc_score(true_labels, scores),
        "ASR": float(np.mean(preds == true_labels))
    }

def run_rare_combo_attack():
    results = {}

    test_path = os.path.join(FOLDER_PATH, TEST_FILE)
    test_df = pd.read_csv(test_path)
    test_df = test_df.dropna(subset=[LABEL_COLUMN])
    test_real = test_df[test_df[LABEL_COLUMN] == "yes"].drop(columns=[LABEL_COLUMN])
    test_labels = test_df[LABEL_COLUMN].tolist()

    for synth_file in SYNTHETIC_FILES:
        synth_path = os.path.join(FOLDER_PATH, synth_file)
        synth_df = pd.read_csv(synth_path)

        shared_cols = [col for col in test_real.columns if col in synth_df.columns]
        test_clean = test_real[shared_cols].copy()
        synth_clean = synth_df[shared_cols].copy()

        print(f"\n🔍 Running Rare Combo Attack on {synth_file} vs {TEST_FILE}")

        scores = []
        for _, row in test_df.iterrows():
            row_df = pd.DataFrame([row.drop(LABEL_COLUMN)])
            score = compute_rare_combo_score(synth_clean, row_df, RARE_COMBOS)
            scores.append(score)

        metrics = evaluate_attack(scores, test_labels)
        results[f"{synth_file} vs {TEST_FILE}"] = metrics

    return results

def print_results(results):
    for key, metrics in results.items():
        print(f"\n==== Results for {key} ====")
        for metric, val in metrics.items():
            print(f"{metric:>10}: {val:.4f}")
        print("-" * 30)

if __name__ == "__main__":
    final_results = run_rare_combo_attack()
    print_results(final_results)



🔍 Running Rare Combo Attack on synthetic_data1.csv vs test_data_wto_outliers.csv

🔍 Running Rare Combo Attack on synthetic_data2.csv vs test_data_wto_outliers.csv

🔍 Running Rare Combo Attack on synthetic_data3.csv vs test_data_wto_outliers.csv

🔍 Running Rare Combo Attack on synthetic_data4.csv vs test_data_wto_outliers.csv

==== Results for synthetic_data1.csv vs test_data_wto_outliers.csv ====
  Accuracy: 0.5000
 Precision: 0.5000
    Recall: 1.0000
        F1: 0.6667
       AUC: 0.5000
       ASR: 0.5000
------------------------------

==== Results for synthetic_data2.csv vs test_data_wto_outliers.csv ====
  Accuracy: 0.5000
 Precision: 0.5000
    Recall: 1.0000
        F1: 0.6667
       AUC: 0.5000
       ASR: 0.5000
------------------------------

==== Results for synthetic_data3.csv vs test_data_wto_outliers.csv ====
  Accuracy: 0.5000
 Precision: 0.5000
    Recall: 1.0000
        F1: 0.6667
       AUC: 0.5000
       ASR: 0.5000
------------------------------

==== Results for 

## Attack 7


In [20]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
)
from scipy.stats import ks_2samp, pearsonr, kendalltau

FOLDER_PATH = "dataset/MIA_SDG_Exercise"
SYNTHETIC_FILES = [f"synthetic_data{i}.csv" for i in range(1, 5)]
TEST_FILE = "test_data_wto_outliers.csv"
LABEL_COLUMN = "is_member"

MARGINAL_COLUMNS = ["lab_test_count", "diagnosis_count", "stay_duration_days", "medication_count"]
JOINT_PAIRS = [("lab_test_count", "diagnosis_count"),
               ("lab_test_count", "procedure_count"),
               ("stay_duration_days", "inpatient_visits")]

# --- Helper Functions --- #
def marginal_divergence(synth_df, test_df):
    scores = []
    for col in MARGINAL_COLUMNS:
        if col not in synth_df.columns or col not in test_df.columns:
            continue
        synth_col = synth_df[col].dropna()
        test_col = test_df[col].dropna()
        if len(synth_col) > 5 and len(test_col) > 5:
            stat, _ = ks_2samp(synth_col, test_col)
            scores.append(stat)
    return np.mean(scores) if scores else 0.0

def joint_divergence(synth_df, test_df, method="pearson"):
    scores = []
    for col1, col2 in JOINT_PAIRS:
        if col1 in synth_df.columns and col2 in synth_df.columns and \
           col1 in test_df.columns and col2 in test_df.columns:
            s1, s2 = synth_df[col1].dropna(), synth_df[col2].dropna()
            t1, t2 = test_df[col1].dropna(), test_df[col2].dropna()

            if len(s1) > 5 and len(s2) > 5 and len(t1) > 5 and len(t2) > 5:
                try:
                    if method == "pearson":
                        corr_s, _ = pearsonr(s1[:min(len(s1), len(s2))], s2[:min(len(s1), len(s2))])
                        corr_t, _ = pearsonr(t1[:min(len(t1), len(t2))], t2[:min(len(t1), len(t2))])
                    elif method == "kendall":
                        corr_s, _ = kendalltau(s1, s2)
                        corr_t, _ = kendalltau(t1, t2)
                    else:
                        continue
                    scores.append(abs(corr_s - corr_t))
                except Exception:
                    continue
    return np.mean(scores) if scores else 0.0

def evaluate_attack(scores, labels, threshold=None):
    scores = np.array(scores)
    true_labels = np.array([1 if l == "yes" else 0 for l in labels])

    if threshold is None:
        threshold = np.median(scores)

    preds = (scores >= threshold).astype(int)

    return {
        "Accuracy": accuracy_score(true_labels, preds),
        "Precision": precision_score(true_labels, preds, zero_division=0),
        "Recall": recall_score(true_labels, preds, zero_division=0),
        "F1": f1_score(true_labels, preds, zero_division=0),
        "AUC": roc_auc_score(true_labels, scores),
        "ASR": float(np.mean(preds == true_labels))
    }

# --- Main Runner --- #
def run_copula_attack():
    results = {}

    test_df = pd.read_csv(os.path.join(FOLDER_PATH, TEST_FILE)).dropna(subset=[LABEL_COLUMN])
    test_labels = test_df[LABEL_COLUMN].tolist()
    test_real = test_df[test_df[LABEL_COLUMN] == "yes"].drop(columns=[LABEL_COLUMN])

    for synth_file in SYNTHETIC_FILES:
        synth_path = os.path.join(FOLDER_PATH, synth_file)
        synth_df = pd.read_csv(synth_path)

        common_cols = [c for c in test_real.columns if c in synth_df.columns]
        test_clean = test_real[common_cols].copy()
        synth_clean = synth_df[common_cols].copy()

        print(f"\n🟢 Gaussian Copula Attack: {synth_file} vs {TEST_FILE}")

        # Per-sample scoring
        scores = []
        for _, row in test_df.iterrows():
            single_row = pd.DataFrame([row.drop(LABEL_COLUMN)])

            m_div = marginal_divergence(synth_clean, single_row)
            j_div = joint_divergence(synth_clean, single_row, method="pearson")
            score = (j_div - m_div)  # High: likely Copula
            scores.append(score)

        metrics = evaluate_attack(scores, test_labels)
        results[f"{synth_file} vs {TEST_FILE}"] = metrics

    return results

def print_results(results):
    for key, metrics in results.items():
        print(f"\n==== Results for {key} ====")
        for metric, val in metrics.items():
            print(f"{metric:>10}: {val:.4f}")
        print("-" * 30)

if __name__ == "__main__":
    res = run_copula_attack()
    print_results(res)



🟢 Gaussian Copula Attack: synthetic_data1.csv vs test_data_wto_outliers.csv

🟢 Gaussian Copula Attack: synthetic_data2.csv vs test_data_wto_outliers.csv

🟢 Gaussian Copula Attack: synthetic_data3.csv vs test_data_wto_outliers.csv

🟢 Gaussian Copula Attack: synthetic_data4.csv vs test_data_wto_outliers.csv

==== Results for synthetic_data1.csv vs test_data_wto_outliers.csv ====
  Accuracy: 0.5000
 Precision: 0.5000
    Recall: 1.0000
        F1: 0.6667
       AUC: 0.5000
       ASR: 0.5000
------------------------------

==== Results for synthetic_data2.csv vs test_data_wto_outliers.csv ====
  Accuracy: 0.5000
 Precision: 0.5000
    Recall: 1.0000
        F1: 0.6667
       AUC: 0.5000
       ASR: 0.5000
------------------------------

==== Results for synthetic_data3.csv vs test_data_wto_outliers.csv ====
  Accuracy: 0.5000
 Precision: 0.5000
    Recall: 1.0000
        F1: 0.6667
       AUC: 0.5000
       ASR: 0.5000
------------------------------

==== Results for synthetic_data4.csv 

## Attack 8

In [21]:
import os
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
)

FOLDER_PATH = "dataset/MIA_SDG_Exercise"
SYNTHETIC_FILES = [f"synthetic_data{i}.csv" for i in range(1, 5)]
TEST_FILE = "test_data_wto_outliers.csv"
LABEL_COLUMN = "is_member"

# Define boundary rules: column -> (min, max)
BOUNDARY_RULES = {
    "inpatient_visits": (0, 10),
    "stay_duration_days": (1, 100),
    "emergency_visits": (0, 10),
    "outpatient_visits": (0, 10),
    "diagnosis_count": (1, 15),
    "lab_test_count": (0, 101),
    "medication_count": (0, 100)
}

# Additional categorical boundary checks (e.g., must have known labels)
REQUIRED_CATEGORIES = {
    "sex": ["Male", "Female"],
    "readmission_status": ["not_readmitted", "readmitted_late", "readmitted_early"],
    "med_change_flag": ["Yes", "No"]
}

# --- Helper --- #
def boundary_violation_score(df):
    score = 0
    n = len(df)

    # Numerical boundaries
    for col, (min_val, max_val) in BOUNDARY_RULES.items():
        if col not in df.columns:
            continue
        series = df[col].dropna()
        violations = ((series < min_val) | (series > max_val)).sum()
        score += violations / max(n, 1)

    # Categorical anomalies
    for col, valid_values in REQUIRED_CATEGORIES.items():
        if col not in df.columns:
            continue
        series = df[col].dropna()
        violations = ~series.isin(valid_values)
        score += violations.sum() / max(n, 1)

    return score

def evaluate_attack(scores, labels, threshold=None):
    scores = np.array(scores)
    true_labels = np.array([1 if l == "yes" else 0 for l in labels])

    if threshold is None:
        threshold = np.median(scores)

    preds = (scores >= threshold).astype(int)

    return {
        "Accuracy": accuracy_score(true_labels, preds),
        "Precision": precision_score(true_labels, preds, zero_division=0),
        "Recall": recall_score(true_labels, preds, zero_division=0),
        "F1": f1_score(true_labels, preds, zero_division=0),
        "AUC": roc_auc_score(true_labels, scores),
        "ASR": float(np.mean(preds == true_labels))
    }

# --- Main Runner --- #
def run_boundary_anomaly_attack():
    results = {}

    test_df = pd.read_csv(os.path.join(FOLDER_PATH, TEST_FILE)).dropna(subset=[LABEL_COLUMN])
    test_labels = test_df[LABEL_COLUMN].tolist()
    test_real = test_df[test_df[LABEL_COLUMN] == "yes"].drop(columns=[LABEL_COLUMN])

    for synth_file in SYNTHETIC_FILES:
        synth_df = pd.read_csv(os.path.join(FOLDER_PATH, synth_file))

        common_cols = [col for col in test_real.columns if col in synth_df.columns]
        synth_clean = synth_df[common_cols].copy()

        print(f"\n🟢 Boundary Violation Attack: {synth_file} vs {TEST_FILE}")

        scores = []
        for _, row in test_df.iterrows():
            single_row = pd.DataFrame([row.drop(LABEL_COLUMN)])
            score = boundary_violation_score(single_row)
            scores.append(score)

        metrics = evaluate_attack(scores, test_labels)
        results[f"{synth_file} vs {TEST_FILE}"] = metrics

    return results

def print_results(results):
    for key, metrics in results.items():
        print(f"\n==== Results for {key} ====")
        for metric, val in metrics.items():
            print(f"{metric:>10}: {val:.4f}")
        print("-" * 30)

if __name__ == "__main__":
    results = run_boundary_anomaly_attack()
    print_results(results)



🟢 Boundary Violation Attack: synthetic_data1.csv vs test_data_wto_outliers.csv

🟢 Boundary Violation Attack: synthetic_data2.csv vs test_data_wto_outliers.csv

🟢 Boundary Violation Attack: synthetic_data3.csv vs test_data_wto_outliers.csv

🟢 Boundary Violation Attack: synthetic_data4.csv vs test_data_wto_outliers.csv

==== Results for synthetic_data1.csv vs test_data_wto_outliers.csv ====
  Accuracy: 0.5000
 Precision: 0.5000
    Recall: 1.0000
        F1: 0.6667
       AUC: 0.5000
       ASR: 0.5000
------------------------------

==== Results for synthetic_data2.csv vs test_data_wto_outliers.csv ====
  Accuracy: 0.5000
 Precision: 0.5000
    Recall: 1.0000
        F1: 0.6667
       AUC: 0.5000
       ASR: 0.5000
------------------------------

==== Results for synthetic_data3.csv vs test_data_wto_outliers.csv ====
  Accuracy: 0.5000
 Precision: 0.5000
    Recall: 1.0000
        F1: 0.6667
       AUC: 0.5000
       ASR: 0.5000
------------------------------

==== Results for syntheti